# Datasets 2 & 3 — Generalization Sets (English and German)

This notebook covers loading, filtering, and exploring the two generalization datasets used to test whether the pipeline transfers beyond the exploratory dataset.

**Source:** QS Lab dataset (October 2025)  
**Dataset 2:** English posts from X and Reddit  
**Dataset 3:** German posts from X and Reddit  
**Labels:** normalized to `TRUE`, `FALSE`, `OTHER`

In [1]:
import sys
import os
from pathlib import Path
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from utils import load_human_posts

PALETTE = {"FALSE": "#D55E00", "TRUE": "#009E73", "OTHER": "#0072B2"}

LABEL_MAP = {
    "TRUE": "TRUE",
    "PARTLY_TRUE": "OTHER",
    "UNPROVEN": "OTHER",
    "MOSTLY_FALSE": "FALSE",
    "FALSE": "FALSE",
    "FALSE ": "FALSE"
}

def is_mostly_url(text):
    cleaned = re.sub(r'http\S+|www\.\S+', '', str(text))
    cleaned = re.sub(r'@\w+', '', cleaned).strip()
    return len(cleaned) < 20

---
## Dataset 2 — English Generalization Set

In [2]:
df_eng = pd.read_csv("data/datasets/generalization-data/english_posts_llm_experiment.csv")
df_eng = df_eng.rename(columns={"item_raw_content": "text"})
df_eng["label"] = df_eng["label"].str.strip().map(LABEL_MAP)
df_eng["X_or_Reddit"] = df_eng["text"].apply(lambda x: "Reddit" if len(x) > 280 else "X")

print(f"Raw: {len(df_eng)} posts")
df_eng = df_eng[~df_eng["text"].apply(is_mostly_url)].reset_index(drop=True)
df_eng = df_eng.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print(f"After filtering: {len(df_eng)} posts")

Raw: 249 posts
After filtering: 215 posts


In [3]:
print("Label distribution:")
print(df_eng["label"].value_counts())
print("\nPlatform distribution:")
print(df_eng["X_or_Reddit"].value_counts())

Label distribution:
label
FALSE    75
OTHER    72
TRUE     68
Name: count, dtype: int64

Platform distribution:
X_or_Reddit
X         147
Reddit     68
Name: count, dtype: int64


In [4]:
vectorizer = CountVectorizer(ngram_range=(1, 3), stop_words="english", max_features=50, min_df=2)
X = vectorizer.fit_transform(df_eng["text"])
freq = pd.Series(X.toarray().sum(axis=0), index=vectorizer.get_feature_names_out()).sort_values(ascending=False)
print(freq.head(30))

https         290
com           148
www           108
https www      82
trump          66
2025           46
org            43
republican     36
israel         34
news           32
10             30
government     30
shutdown       24
en             23
people         22
year           19
sex            19
federal        17
gaza           17
like           17
01             17
2025 10        16
world          16
just           15
state          15
old            15
russian        14
says           14
wikipedia      14
year old       14
dtype: int64


In [5]:
for name, pattern in [
    ("Shutdown", "shutdown|federal|funding|continuing resolution"),
    ("Israel", "israel|gaza|israeli|jewish|forces|palestine|idf"),
    ("Covid", "covid|vaccine"),
]:
    cluster = df_eng[df_eng["text"].str.contains(pattern, case=False)]
    print(f"\n--- {name} (n={len(cluster)}) ---")
    print(cluster["label"].value_counts())
    print(f"FALSE rate: {(cluster['label'] == 'FALSE').mean():.2%}")


--- Shutdown (n=26) ---
label
TRUE     13
OTHER     9
FALSE     4
Name: count, dtype: int64
FALSE rate: 15.38%

--- Israel (n=39) ---
label
FALSE    15
OTHER    15
TRUE      9
Name: count, dtype: int64
FALSE rate: 38.46%

--- Covid (n=4) ---
label
FALSE    3
TRUE     1
Name: count, dtype: int64
FALSE rate: 75.00%


In [6]:
# Confirm via utility function (applies URL normalization and deduplication)
human_posts_eng = load_human_posts(2)
print(f"Loaded {len(human_posts_eng)} English posts via load_human_posts(2)")
human_posts_eng.head()

Loaded 215 English posts via load_human_posts(2)


,text,label
0,@atrupar Oh come on\nIt was always the plan\nTo take 2 million jobs away\nDOGE quit early\nWake up USA! \nTrump considers ypu + your job to be debris _URL_,OTHER
1,The Erika Kirk Israeli Honey Trap Rabbit Hole Gets Deeper &amp; More Sinister… _URL_ via @YouTube @realnikohouse --_URL_,FALSE
2,@disclosetv Just wheel out that magic money machine @elonmusk told us about. Wait never mind…that’s for funding war. _URL_,FALSE
3,"Drunk with power': Author tells how Chief Justice John Roberts 'corrupted' Supreme Court \n_URL_\n… \n\nSix Conservative Justices are bought and paid for, easily Corrupted!",FALSE
4,"MACC wins forfeiture of RM169m cash linked to ex-PM Ismail Sabri \n_URL_ can someone, please confirm if a crime has been committed? Is that 169 his money that he is allowing govt to wallop? Is he not guilty for not declaring from where this 169 came from.",TRUE


---
## Dataset 3 — German Generalization Set

In [7]:
df_ger = pd.read_csv("data/datasets/generalization-data/german_posts_llm_experiment.csv")
df_ger = df_ger.rename(columns={"item_raw_content": "text", "final_normalized": "label"})[[ "text", "label"]]
df_ger["label"] = df_ger["label"].str.strip().map(LABEL_MAP)
df_ger["X_or_Reddit"] = df_ger["text"].apply(lambda x: "Reddit" if len(x) > 280 else "X")

print(f"Raw: {len(df_ger)} posts")
df_ger = df_ger[~df_ger["text"].apply(is_mostly_url)].reset_index(drop=True)
df_ger = df_ger.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print(f"After filtering: {len(df_ger)} posts")

Raw: 249 posts
After filtering: 214 posts


In [8]:
print("Label distribution:")
print(df_ger["label"].value_counts())
print("\nPlatform distribution:")
print(df_ger["X_or_Reddit"].value_counts())

Label distribution:
label
TRUE     106
OTHER     56
FALSE     52
Name: count, dtype: int64

Platform distribution:
X_or_Reddit
X         108
Reddit    106
Name: count, dtype: int64


In [9]:
for name, pattern in [
    ("Israel", "israel|gaza|israeli|juden|hamas|idf|greta"),
    ("Oktoberfest", "oktoberfest|münchen|bombe|bombedrohung|munich"),
    ("Ukraine", "ukraine|russland|nato|selenskyj|zelenski|kreml"),
]:
    cluster = df_ger[df_ger["text"].str.contains(pattern, case=False)]
    print(f"\n--- {name} (n={len(cluster)}) ---")
    print(cluster["label"].value_counts())
    print(f"FALSE rate: {(cluster['label'] == 'FALSE').mean():.2%}")


--- Israel (n=40) ---
label
TRUE     20
FALSE    10
OTHER    10
Name: count, dtype: int64
FALSE rate: 25.00%

--- Oktoberfest (n=13) ---
label
TRUE     7
FALSE    4
OTHER    2
Name: count, dtype: int64
FALSE rate: 30.77%

--- Ukraine (n=29) ---
label
TRUE     21
FALSE     5
OTHER     3
Name: count, dtype: int64
FALSE rate: 17.24%


In [10]:
human_posts_ger = load_human_posts(3)
print(f"Loaded {len(human_posts_ger)} German posts via load_human_posts(3)")
human_posts_ger.head()

Loaded 214 German posts via load_human_posts(3)


,text,label
0,USA: Richter bremst US-Regierung bei Vorgehen gegen Gaza-Protest\n_URL_\nSpiegel Ausland GRBsB Skandal Menschenrechte Nahost\nDie US-Regierung droht ausländischen Studierenden wegen ihrer Teilnahme an Gazaprotesten mit Abschiebungen. Nun bremst ein Bundesri… _URL_ --_URL_,TRUE
1,"🚀 Die Ära günstiger Lebensmittel ist vorbei: +37% seit 2020! 📈\n\nWährend die EZB von 3,2% Nahrungsmittelinflation spricht, zeigt Dein Kassenzettel die bittere Wahrheit. 🛒💸\n\nDie Statistik-Tricks:\n• Hedonische Anpassungen drücken künstlich die Zahlen\n• Shrinkflation bleibt oft unerkannt\n• Methodische Unterschiede verschleiern bis zu 0,6%\n\nFür Dich als Investor bedeutet das: Die reale Inflation frisst mehr Kaufkraft als offiziell zugegeben. Zeit, Deine Strategie zu überdenken! 💡\n\n👉 _URL_ --_URL_",OTHER
2,">Allerdings entschied der VGH auch, dass der Verfassungsschutz die Beobachtung der AfD nicht hätte öffentlich bekanntgeben dürfen. Auch damit bestätigte er die Entscheidung des Verwaltungsgerichts Wiesbaden: Das hatte geurteilt, dass eine öffentliche Bekanntgabe der Beobachtung eine gesetzliche Ermächtigungsgrundlage brauche. Denn es habe Auswirkungen auf das Recht der AfD, gleichberechtigt am Prozess der Meinungs- und Willensbildung des Volkes teilzunehmen.\n>\n>Auch das hessische Innenministerium hätte nicht per Pressemitteilung über die Beobachtung der AfD informieren dürfen, entschied der VGH. Er wies eine Beschwerde des Ministeriums gegen die entsprechende Entscheidung des Verwaltungsgerichts Wiesbaden zurück.\n>\n>Der VGH wies eine weitere Beschwerde der AfD zurück, die sich auf Äußerungen von Ministerpräsident Boris Rhein (CDU) bezog: Rhein hatte die AfD 2022 als ""im Kern radikal und gefährlich"" bezeichnet. Dagegen wehrte sich die Partei.\n>\n>Der VGH erklärte sich für nicht zuständig für diese Beschwerde, da der Ministerpräsident als Verfassungsorgan gehandelt habe. Somit gehe es hier nicht um eine verwaltungsrechtliche, sondern um eine verfassungsrechtliche Angelegenheit. Zuständig sei der Hessische Staatsgerichtshof.\n>\n>Die AfD kündigte an, sich weiter gerichtlich gegen die Einstufung als Verdachtsfall zu wehren. ""Nach der Entscheidung im Eilverfahren geht es nun im Hauptsacheverfahren vor das Verwaltungsgericht Wiesbaden"", heißt es in einer Pressemitteilung.\n>\n>Damit bezog sich die Partei darauf, dass der Hessische Verwaltungsgerichtshof bislang lediglich in einem Eilverfahren entschieden hat - unanfechtbar und höchstens mit der Möglichkeit einer Verfassungsbeschwerde.\n>\n>Der hessische Innenminister Roman Poseck (CDU) begrüßte die neue Gerichtsentscheidung: ""Sie bringt endlich Rechtsklarheit in einer für unsere Demokratie zentralen Frage."" Der Verwaltungsgerichtshof habe die Auffassung des Verwaltungsgerichts Wiesbaden bestätigt, ""dass die AfD in Hessen gegen die Menschenwürde und das Demokratieprinzip verstößt"". \n>\n>Poseck kündigte an: ""Das Landesamt für Verfassungsschutz wird den Beschluss auswerten und die daraus folgenden Konsequenzen für den weiteren Umgang mit der hessischen AfD ziehen."" Der Innenminister, einst höchster Richter Hessens, hatte vor rund drei Wochen das lange Verfahren um die AfD-Einstufung kritisiert. Das Land habe einen ""Maulkorb"" erhalten, dem Verfassungsschutz seien ""Fesseln angelegt"", hatte Poseck gesagt. Deshalb tauche die AfD in dem im September veröffentlichten Bericht des LfV auch nicht auf.\n>\n>Laut der Grünen-Fraktion im Landtag unterstreicht der neue Gerichtsbeschluss, dass die AfD längst keine Protestpartei mehr sei: ""Sie wendet sich gegen unsere Freiheit, unsere Demokratie und gegen die Menschenwürde. Deshalb wird sie völlig zu Recht vom Verfassungsschutz beobachtet.""\n>\n>Die FDP-Fraktion befand: ""Die hessische AfD hat es bis heute nicht geschafft, sich von Rechtsextremisten wie Björn Höcke, dem sogenannten Flügel und der Identitären Bewegung zu distanzieren - im Gegenteil: Flügel und Freunde der Identitären Bewegung sind Teil der Fraktion.""\n\n_URL_",TRUE
